# Exp 6 — Inference-gated Budget Router

Tests the router on the GSM8K test set using the filtered distillation checkpoint (Exp 4).

The router scores each prompt's complexity with one forward pass, then routes to:
- **easy** profile: `max_new_tokens=128` (low cost)
- **hard** profile: `max_new_tokens=300` (full CoT budget)

Sweeps 10 thresholds between p10–p90 of the complexity score distribution to build a Pareto curve (accuracy vs token cost).

**Prerequisite:** checkpoint is downloaded automatically from Google Drive in cell 3.

In [ ]:
import os, sys

if not os.path.exists('/kaggle/working/SC4001-Group133/.git'):
    os.system('git clone https://github.com/TIFFMUN/SC4001-Group133.git /kaggle/working/SC4001-Group133')

os.chdir('/kaggle/working/SC4001-Group133')
os.system('git fetch origin main')
os.system('git reset --hard origin/main')

import importlib
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]
importlib.invalidate_caches()

os.system('pip install -r requirements.txt -q')
os.system('pip install -e . -q')

import subprocess
print(subprocess.check_output(['git', 'log', '--oneline', '-1']).decode().strip())

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Download Exp 4 checkpoint from Google Drive
import os

CHECKPOINT_PATH = 'results/checkpoints/filtered'
MAX_SAMPLES = 100   # increase to 200 or None for full eval
EXPERIMENT_NAME = 'exp6_router'

os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.system('pip install gdown -q')

# Download checkpoint folder from Drive
os.system(f'gdown --folder https://drive.google.com/drive/folders/1lZFRVYXABetcmfMc6w4QoIuHMx4R3l1Z -O {CHECKPOINT_PATH} --remaining-ok')

if os.path.exists(CHECKPOINT_PATH) and os.listdir(CHECKPOINT_PATH):
    print(f'Checkpoint ready: {os.listdir(CHECKPOINT_PATH)}')
else:
    print('WARNING: checkpoint download may have failed — will use base model')

In [ ]:
# Exp 1e — format-reinforced prompt, 500 samples, base model
MAX_SAMPLES = 500

import json
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

from src.baselines.cot_prompting import build_format_reinforced_prompt
from src.distillation.filter_traces import extract_answer, answers_match
from src.evaluation.metrics import compute_latency, compute_summary_stats, save_results, print_summary

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()

test_data = load_dataset("openai/gsm8k", "main")["test"].select(range(MAX_SAMPLES))
name = "exp1e_format_reinforced_500"

results = []
with torch.no_grad():
    for i, sample in enumerate(tqdm(test_data, desc=name)):
        prompt = build_format_reinforced_prompt(sample["question"], tokenizer)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        latency_ms, output_ids = compute_latency(
            model=model, inputs=inputs, max_new_tokens=256, do_sample=False
        )
        generated_ids = output_ids[0][input_len:]
        prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)
        predicted_answer = extract_answer(prediction)
        is_correct = answers_match(predicted_answer, sample["answer"])
        results.append({
            "question": sample["question"],
            "ground_truth": sample["answer"],
            "prediction": prediction,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct,
            "latency_ms": latency_ms,
            "token_count": len(generated_ids),
        })
        if (i + 1) % 100 == 0:
            save_results(results, f"results/{name}_gsm8k_partial.csv")

stats = compute_summary_stats(results)
print_summary(name, stats)
save_results(results, f"results/{name}_gsm8k.csv")
stats_path = Path("results/summary_results") / f"{name}_gsm8k_summary.json"
stats_path.parent.mkdir(parents=True, exist_ok=True)
with open(stats_path, "w") as f:
    json.dump({"experiment": name, **stats}, f, indent=2)

In [ ]:
from src.evaluation.eval_router import run_router_sweep

sweep_df = run_router_sweep(
    experiment_name=EXPERIMENT_NAME,
    max_samples=MAX_SAMPLES,
    checkpoint_override=CHECKPOINT_PATH,
)

print(sweep_df[['threshold', 'accuracy', 'mean_token_count', 'pct_easy', 'pareto_optimal']].to_string(index=False))

In [ ]:
# Pareto-optimal points only
pareto = sweep_df[sweep_df['pareto_optimal']]
print('=== Pareto-optimal thresholds ===')
print(pareto[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/router_results', 'zip', 'results')
print('Saved to Output tab: router_results.zip')